<a href="https://colab.research.google.com/github/edgaralbasanz/ealba-xatbot/blob/main/XatBot2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!lsof -ti:5000 | xargs kill -9
from pyngrok import ngrok
ngrok.kill()


Usage:
 kill [options] <pid> [...]

Options:
 <pid> [...]            send signal to every <pid> listed
 -<signal>, -s, --signal <signal>
                        specify the <signal> to be sent
 -q, --queue <value>    integer value to be sent with the signal
 -l, --list=[<signal>]  list all signal names, or convert one to a name
 -L, --table            list all signal names in a nice table

 -h, --help     display this help and exit
 -V, --version  output version information and exit

For more details see kill(1).


In [ ]:
from google.colab import userdata
import google.generativeai as genai
import requests
from bs4 import BeautifulSoup
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok
import re
from urllib.parse import urljoin, urlparse
import time

# =================== SECRETS ===================
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
genai.configure(api_key=GEMINI_API_KEY)

# =================== MODEL ===================
model = genai.GenerativeModel('gemini-2.5-flash-lite')

app = Flask(__name__)
CORS(app)

# =================== CRAWLER ===================
print("🚀 Iniciant CRAWLER...")

BASE_URL = "https://ealba.inscastellbisbal.net"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

visited = set()
to_visit = [BASE_URL]
all_content = []

start_time = time.time()
max_time = 150

while to_visit and (time.time() - start_time) < max_time:
    current_url = to_visit.pop(0)
    if current_url in visited: continue

    print(f"   📄 {current_url}")
    visited.add(current_url)

    try:
        r = requests.get(current_url, headers=headers, timeout=10)
        if r.status_code != 200: continue

        soup = BeautifulSoup(r.text, 'html.parser')

        texts = []
        for tag in soup.find_all(['h1','h2','h3','p','li','strong','em']):
            text = tag.get_text(strip=True)
            if len(text) > 25:
                texts.append(text)

        for img in soup.find_all('img'):
            if img.get('alt'):
                texts.append(f"Imatge: {img.get('alt')}")

        if texts:
            all_content.append(" | ".join(texts))

        for link in soup.find_all('a', href=True):
            full_url = urljoin(current_url, link['href'])
            if urlparse(full_url).netloc in ["", "ealba.inscastellbisbal.net"]:
                if full_url not in visited and full_url not in to_visit and not full_url.endswith(('.pdf','.jpg','.png')):
                    to_visit.append(full_url)
    except:
        continue
    time.sleep(0.4)

GLOBAL_CONTEXT = " ".join(all_content)
GLOBAL_CONTEXT = re.sub(r'\s+', ' ', GLOBAL_CONTEXT)

print(f"\n✅ CRAWLER FINALITZAT! Pàgines: {len(visited)}")

# =================== CLEAN RESPONSE ===================
def clean_response(text):
    text = re.sub(r'\*\*(.+?)\*\*', r'\1', text)
    text = re.sub(r'\*(.+?)\*', r'\1', text)
    lines = text.split('\n')
    if len(lines) > 12:
        text = '\n'.join(lines[:12]) + "\n..."
    return text.strip()

# =================== CHAT ===================
@app.route('/chat', methods=['POST'])
def chat():
    try:
        user_message = request.json.get('message', '') if request.json else ''

        prompt = f"""Ets l'assistent oficial d'Edgar Alba.
Respon de forma clara, directa i concisa (màxim 10-12 línies).
Utilitza paràgrafs curts. No utilitzis Markdown.
Sigues professional però amable.

Informació del lloc web:
{GLOBAL_CONTEXT[:16500]}

Pregunta: {user_message}"""

        response = model.generate_content(prompt)
        clean_reply = clean_response(response.text)

        return jsonify({"reply": clean_reply})

    except Exception as e:
        print(f"Error: {e}")
        return jsonify({"reply": "Ho sento, hi ha hagut un error. Torna-ho a provar."})

# ===================== INICIAR =====================
public_url = ngrok.connect(5000)
print("\n" + "="*70)
print("✅ XATBOT d'Edgar Alba EN FUNCIONAMENT")
print(f"🔗 URL: {public_url}")
print(f"📊 Pàgines carregades: {len(visited)}")
print("="*70)

app.run(port=5000)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


🚀 Iniciant CRAWLER...
   📄 https://ealba.inscastellbisbal.net
   📄 https://ealba.inscastellbisbal.net#wp--skip-link--target
   📄 https://ealba.inscastellbisbal.net/
   📄 https://ealba.inscastellbisbal.net/apunts/
   📄 https://ealba.inscastellbisbal.net/politica-de-privadesa/
   📄 https://ealba.inscastellbisbal.net/politica-de-privadesa/privacy-policy/
   📄 https://ealba.inscastellbisbal.net/reptes/
   📄 https://ealba.inscastellbisbal.net/reptes/repte-1-1/
   📄 https://ealba.inscastellbisbal.net/reptes/repte-1-1/espai-de-treball-al-nuvol/
   📄 https://ealba.inscastellbisbal.net/reptes/repte-1-1/correu-professional/
   📄 https://ealba.inscastellbisbal.net/reptes/repte-1-1/correu-corporatiu-del-centre/
   📄 https://ealba.inscastellbisbal.net/reptes/repte-1-1/portafolis-digital/
   📄 https://ealba.inscastellbisbal.net/reptes/repte-1-1/plataformes-web/
   📄 https://ealba.inscastellbisbal.net/reptes/repte-1-1/agenda-electronica/
   📄 https://ealba.inscastellbisbal.net/reptes/repte-1-1/llibre

INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [08/Jun/2026 10:06:58] "OPTIONS /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/Jun/2026 10:07:00] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/Jun/2026 10:07:18] "OPTIONS /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/Jun/2026 10:07:20] "POST /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/Jun/2026 10:07:32] "OPTIONS /chat HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [08/Jun/2026 10:07:33] "POST /chat HTTP/1.1" 200 -
